# Taller II · Evaluación Sumativa 3
**MCDI502 · Gestión de Datos y Tecnologías Big Data · Grupo 4**

Integrantes: Felipe Díaz Toro · Ninoska Yévenes Hernández · Guillermo Subiabre Chacón · Pablo Ríos Passteni

Este notebook sigue la estructura del enunciado: 1 Configuración · 2 Ingesta (cuatro DataFrames de 100 registros) · 3 Preprocesamiento · 4 Integración · 5 Consultas con Spark SQL.

| Sección | Responsable |
|---|---|
| 1. Configuración del entorno | Felipe |
| 2.1 OpenMeteo, 3.2 `condicion_clima`, consultas C1, C2, C3, C7 | Ninoska |
| 2.2 AQICN, consultas C4, C5 | Guillermo |
| 2.3 MongoDB, 2.4 MySQL, 3.1 `altura_metros`, consulta C6 | Pablo |
| 4. Integración (`df_unificado`) y revisión final | Felipe |

**Reglas del grupo:** cada uno trabaja en su rama y toca solo su sección; antes de entregar se reinicia el kernel y se ejecuta todo de punta a punta. Los secretos (token de AQICN, URI de MongoDB) van en el archivo `.env` de la raíz del repo, que **no se sube a GitHub**.

# 1. Configuración inicial del entorno
*Responsable: Felipe.* El notebook se ejecuta en VS Code con el entorno de conda del grupo (`grupo4_mcdi502`), que ya trae Java 17 y PySpark. Se crean las variables de entorno, la `SparkSession` y el `SparkContext`, y se instalan las librerías para las fuentes de datos (requests, pandas, pymongo, mysql-connector-python). El conector de Spark para MongoDB lo descarga Spark al iniciar la sesión.

In [ ]:
# 1.1 Librerías para las fuentes de datos (si ya están instaladas, pip no hace nada)
%pip install -q requests pandas pymongo mysql-connector-python

In [ ]:
# 1.2 Variables de entorno: JAVA_HOME apunta al Java del entorno de conda
import os, sys
os.environ["JAVA_HOME"] = os.path.join(sys.prefix, "lib", "jvm")
print("JAVA_HOME =", os.environ["JAVA_HOME"])

In [ ]:
# 1.3 SparkSession y SparkContext
# El conector de MongoDB (mismo del notebook base del docente) se descarga desde Maven al crear la sesión
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Taller2 Grupo4")
         .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1")
         .getOrCreate())
sc = spark.sparkContext
print("Spark", spark.version, "| aplicación:", sc.appName)

In [ ]:
# 1.4 Configuración de fuentes. Los secretos van en un archivo .env en la raíz del repo (está en .gitignore)
# Formato del .env, una línea por valor, sin comillas:
#   AQICN_TOKEN=...
#   MONGO_URI=...
def cargar_env(ruta_env="../.env"):
    valores = {}
    if os.path.exists(ruta_env):
        for linea in open(ruta_env, encoding="utf-8"):
            if "=" in linea and not linea.strip().startswith("#"):
                k, v = linea.strip().split("=", 1)
                valores[k.strip()] = v.strip()
    return valores

_env = cargar_env()
AQICN_TOKEN = _env.get("AQICN_TOKEN", "PEGAR_TOKEN_PROPIO")   # token gratuito de https://aqicn.org/data-platform/token/
AQICN_CIUDAD = "santiago"                                      # ciudad/región del grupo
MONGO_URI = _env.get("MONGO_URI", "PEGAR_URI_DEL_DOCENTE")     # está en el notebook base BaseEjemploMongoDB
MONGO_DB, MONGO_COL = "espdb", "datos"                         # valores del notebook base
MYSQL_URL = "http://mgtt.comunidadingenieria.cl/obtener_datos.php"
N_REGISTROS = 100                                              # el enunciado pide 100 por DataFrame

# 2. Ingesta de datos y creación de DataFrames (ID3.1)
Cada fuente se consulta, se deja en exactamente 100 registros y se convierte a un DataFrame de Spark.

## 2.1 OpenMeteo → `df_openmeteo`  *(Ninoska)*
Se pide una serie horaria de los últimos días, que entrega más de 100 registros de una sola vez y con fecha, lo que sirve para las consultas C2 y C3. Las coordenadas son las del notebook base (Viña del Mar).

In [ ]:
import requests, pandas as pd

LAT, LON = -33.02457, -71.55183
VARS = ["temperature_2m", "relative_humidity_2m", "precipitation",
        "wind_speed_10m", "wind_direction_10m", "uv_index"]
url = ("https://api.open-meteo.com/v1/forecast"
       f"?latitude={LAT}&longitude={LON}&hourly={','.join(VARS)}"
       "&past_days=7&forecast_days=1&timezone=America/Santiago")
h = requests.get(url, timeout=30).json()["hourly"]
pdf_om = pd.DataFrame(h).rename(columns={"time": "tiempo"})
pdf_om["tiempo"] = pd.to_datetime(pdf_om["tiempo"])
# últimos 100 registros que ya ocurrieron (no pronóstico)
pdf_om = pdf_om[pdf_om["tiempo"] <= pd.Timestamp.now()].tail(N_REGISTROS).reset_index(drop=True)

df_openmeteo = spark.createDataFrame(pdf_om)
print("Registros:", df_openmeteo.count())
df_openmeteo.show(5, truncate=False)

## 2.2 AQICN → `df_aqicn`  *(Guillermo)*
El token `demo` del notebook base no corresponde a la región del grupo; se usa el token propio de la celda 1.4.

In [ ]:
import time
rows = []
url = f"http://api.waqi.info/feed/{AQICN_CIUDAD}/?token={AQICN_TOKEN}"
while len(rows) < N_REGISTROS:
    d = requests.get(url, timeout=30).json()
    if d.get("status") == "ok":
        iaqi = d["data"]["iaqi"]
        rows.append({
            "tiempo": d["data"]["time"]["s"],
            "PM2_5": float(iaqi.get("pm25", {}).get("v", "nan")),
            "PM10": float(iaqi.get("pm10", {}).get("v", "nan")),
            "O3":   float(iaqi.get("o3",   {}).get("v", "nan")),
            "NO2":  float(iaqi.get("no2",  {}).get("v", "nan")),
            "SO2":  float(iaqi.get("so2",  {}).get("v", "nan")),
        })
    else:
        print("Respuesta no válida:", d)
        break
    time.sleep(1)

df_aqicn = spark.createDataFrame(pd.DataFrame(rows))
print("Registros:", df_aqicn.count())
df_aqicn.show(5, truncate=False)

## 2.3 MongoDB → `df_mongo_bpm`  *(Pablo)*
Se lee la colección del docente con el conector de Spark. Las variables a capturar son `alturaft`, presión atmosférica y monóxido de carbono; **ajustar los nombres de las columnas a los que muestre `printSchema()`**.

In [ ]:
df_mongo_raw = (spark.read.format("com.mongodb.spark.sql.DefaultSource")
                .option("uri", MONGO_URI)
                .option("database", MONGO_DB)
                .option("collection", MONGO_COL)
                .load())
df_mongo_raw.printSchema()

In [ ]:
# TODO (Pablo): reemplazar los nombres de la lista por las columnas reales vistas arriba
cols_mongo = ["alturaft"]   # añadir presión atmosférica y monóxido de carbono
df_mongo_bpm = df_mongo_raw.select(*cols_mongo).limit(N_REGISTROS)
print("Registros:", df_mongo_bpm.count())
df_mongo_bpm.show(5, truncate=False)

## 2.4 MySQL → `df_mysql_dht`  *(Pablo)*
La base MySQL se consulta a través de la API PHP, que devuelve JSON con `id`, `temperatura`, `humedad` y `fecha`.

In [ ]:
data = requests.get(MYSQL_URL, timeout=30).json()
pdf_my = pd.DataFrame(data)
pdf_my["temperatura_interior"] = pdf_my["temperatura"].astype(float)
pdf_my["humedad"] = pdf_my["humedad"].astype(float)
pdf_my = pdf_my[["id", "fecha", "temperatura_interior", "humedad"]].tail(N_REGISTROS).reset_index(drop=True)

df_mysql_dht = spark.createDataFrame(pdf_my)
print("Registros:", df_mysql_dht.count())
df_mysql_dht.show(5, truncate=False)

# 3. Preprocesamiento y transformación de datos (RA4, ID4.2)

## 3.1 Columna `altura_metros` en `df_mongo_bpm`  *(Pablo)*
1 pie = 0,3048 metros.

In [ ]:
from pyspark.sql import functions as F
df_mongo_bpm = df_mongo_bpm.withColumn("altura_metros", F.round(F.col("alturaft") * 0.3048, 4))
df_mongo_bpm.select("alturaft", "altura_metros").show(5)

## 3.2 Columna `condicion_clima` en `df_openmeteo`  *(Ninoska)*
Lógica del enunciado: calor y humedad (T > 30 y H > 70), calor seco (T > 30 y H ≤ 70), clima moderado (T ≤ 30) y, en otro caso (datos nulos), condición desconocida.

In [ ]:
t, hum = F.col("temperature_2m"), F.col("relative_humidity_2m")
df_openmeteo = df_openmeteo.withColumn(
    "condicion_clima",
    F.when((t > 30) & (hum > 70), "CALOR Y HUMEDAD")
     .when((t > 30) & (hum <= 70), "CALOR SECO")
     .when(t <= 30, "CLIMA MODERADO")
     .otherwise("CONDICION DESCONOCIDA"))
df_openmeteo.groupBy("condicion_clima").count().show()

# 4. Integración de datos (RA3)  *(Felipe)*
**Estrategia:** las cuatro fuentes son heterogéneas y no comparten una clave natural (cada una tiene su propia marca de tiempo y su propio origen). Se asume que el registro *n* de cada fuente corresponde al mismo instante de captura, por lo que se unen **por índice de fila** (join por un número consecutivo). Es una decisión simplificadora y queda documentada: no se afirma una correspondencia temporal real entre sensores y APIs. Como las cuatro tablas tienen 100 registros, no se pierden filas.

In [ ]:
from pyspark.sql.window import Window

def con_indice(df, orden):
    w = Window.orderBy(F.monotonically_increasing_id())
    return df.withColumn("idx", F.row_number().over(w))

om = con_indice(df_openmeteo, None).select(
    "idx", F.col("tiempo").alias("tiempo_openmeteo"), "temperature_2m", "relative_humidity_2m",
    "precipitation", "wind_speed_10m", "wind_direction_10m", "uv_index", "condicion_clima")
aq = con_indice(df_aqicn, None).select("idx", "PM2_5", "PM10", "O3", "NO2", "SO2")
mo = con_indice(df_mongo_bpm, None)          # ajustar el select a las columnas finales de Mongo
my = con_indice(df_mysql_dht, None).select("idx", "temperatura_interior", "humedad")

df_unificado = (om.join(aq, "idx").join(mo, "idx").join(my, "idx")).orderBy("idx")
print("Filas:", df_unificado.count(), "| Columnas:", len(df_unificado.columns))
df_unificado.show(5, truncate=False)

# 5. Consultas y extracción de datos (ID3.1, ID4.1)
Primero se crea una tabla temporal por DataFrame con Spark SQL.

In [ ]:
df_openmeteo.createOrReplaceTempView("openmeteo")
df_aqicn.createOrReplaceTempView("aqicn")
df_mongo_bpm.createOrReplaceTempView("mongo_bpm")
df_mysql_dht.createOrReplaceTempView("mysql_dht")
print(spark.catalog.listTables())

**C1** · 10 registros con mayor temperatura  *(Ninoska)*

In [ ]:
spark.sql("SELECT * FROM openmeteo ORDER BY temperature_2m DESC LIMIT 10").show(truncate=False)

**C2** · Promedio de humedad relativa para un mes y día específicos  *(Ninoska)*. Elegir un día que exista en los datos (se muestran los disponibles).

In [ ]:
spark.sql("SELECT DISTINCT month(tiempo) AS mes, day(tiempo) AS dia FROM openmeteo ORDER BY 1, 2").show(20)
MES, DIA = 11, 5   # TODO: ajustar a un día presente en los datos
spark.sql(f"""SELECT {MES} AS mes, {DIA} AS dia, ROUND(AVG(relative_humidity_2m), 2) AS humedad_promedio
              FROM openmeteo WHERE month(tiempo) = {MES} AND day(tiempo) = {DIA}""").show()

**C3** · Registros con precipitación > 2 mm durante la última semana  *(Ninoska)*

In [ ]:
spark.sql("""SELECT COUNT(*) AS registros_sobre_2mm FROM openmeteo
              WHERE precipitation > 2 AND tiempo >= date_sub(current_date(), 7)""").show()

**C4** · Promedio de PM2.5, PM10, O3, NO2 y SO2  *(Guillermo)*

In [ ]:
spark.sql("SELECT AVG(PM2_5) AS pm25, AVG(PM10) AS pm10, AVG(O3) AS o3, AVG(NO2) AS no2, AVG(SO2) AS so2 FROM aqicn").show()

**C5** · Registros con NO2 superior a 4  *(Guillermo)*

In [ ]:
spark.sql("SELECT COUNT(*) AS registros_no2_mayor_4 FROM aqicn WHERE NO2 > 4").show()

**C6** · Primeros 10 registros de `df_mongo_bpm` con `altura_metros`  *(Pablo)*

In [ ]:
spark.sql("SELECT * FROM mongo_bpm LIMIT 10").show(truncate=False)

**C7** · Primeros 10 registros de `df_openmeteo` con `condicion_clima`  *(Ninoska)*

In [ ]:
spark.sql("SELECT * FROM openmeteo LIMIT 10").show(truncate=False)

# 6. Notas para el video
- Qué conexión usó cada fuente y por qué: OpenMeteo (serie horaria para llegar a 100 registros con fecha), AQICN (token propio), MongoDB (conector de Spark), MySQL (API PHP).
- Por qué la integración es por índice y qué limitación tiene.
- Qué resultado dio cada consulta (C1 a C7) y qué significa.